In [1]:
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

### Constants

In [3]:
str_image_name = 'genxi-pull-payloads'
int_iteration = 1
str_instance = 'm5.xlarge'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

int_vcpu: 4
int_memory_gb: 16
int_memory_mebibytes: 15259


### Create compute environment

In [4]:
# initialize class
cls_client = boto3.client('batch')

In [5]:
# get role
try:
    str_role = get_execution_role()
except:
    ! pip install --upgrade boto3
    str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [6]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '163',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 30 Apr 2024 17:12:44 GMT',
                                      'x-amz-apigw-id': 'XDL18EaRPHcEB3w=',
                                      'x-amzn-requestid': '5fa9e9e7-8685-45ad-b24b-cd9386c9e927',
                                      'x-amzn-trace-id': 'Root=1-6631268c-3b68eb81708904c17df3cf8b'},
                      'HTTPStatusCode': 200,
                      'RequestId': '5fa9e9e7-8685-45ad-b24b-cd9386c9e927',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create Job Queue

In [7]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '137',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 30 Apr 2024 17:12:52 GMT',
                                      'x-amz-apigw-id': 'XDL3RG4zvHcEJ9g=',
                                      'x-amzn-requestid': '3cf92992-38b8-4352-ba19-d2fea56a0541',
                                      'x-amzn-trace-id': 'Root=1-66312694-1897c8690a81f2257bfd3378'},
                      'HTTPStatusCode': 200,
                      'RequestId': '3cf92992-38b8-4352-ba19-d2fea56a0541',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [8]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '171',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 30 Apr 2024 17:12:52 GMT',
                                      'x-amz-apigw-id': 'XDL3SFWGPHcELmQ=',
                                      'x-amzn-requestid': 'bb22cfa7-3f7a-4e44-bc51-507f44172365',
                                      'x-amzn-trace-id': 'Root=1-66312694-35a2016c7de49df5280cddee'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'bb22cfa7-3f7a-4e44-bc51-507f44172365',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [9]:
# # submit a job (only for testing)
# while True:
#     try:
#         str_job_name = f'job-name-{str_image_name}-{int_iteration}'
#         response = cls_client.submit_job(
#             jobDefinition=str_job_definition,
#             jobQueue=str_job_queue_name,
#             jobName=str_job_name,
#         )
#         pprint(response)
#         break
#     except:
#         time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '180',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 30 Apr 2024 17:12:55 GMT',
                                      'x-amz-apigw-id': 'XDL3qGkuvHcEmEA=',
                                      'x-amzn-requestid': '0a47e490-801f-4639-901a-3e144ed107ba',
                                      'x-amzn-trace-id': 'Root=1-66312697-45b13fd02e11414a0676e488'},
                      'HTTPStatusCode': 200,
                      'RequestId': '0a47e490-801f-4639-901a-3e144ed107ba',
                      'RetryAttempts': 0},
 'jobArn': 'arn:aws:batch:us-west-2:83669

### Show arns

In [10]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxi-pull-payloads-1
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxi-pull-payloads-1:1
